# Brain MRI tumour segmentation: guided walkthrough

End-to-end tour of the `brainseg` package on the LGG dataset. Run `make data` first (or point `ROOT` at your copy).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from brainseg.data import find_pairs, patient_split, load_arrays, make_dataset
from brainseg.viz import overlay
ROOT = "../Dataset/lgg-mri-segmentation"

## 1. Inspect the data and split by patient

In [ ]:
df = find_pairs(ROOT)
train, val, test = patient_split(df, 0.15, 0.15, seed=42)
print(f"{df.patient.nunique()} patients | train/val/test slices: {len(train)}/{len(val)}/{len(test)}")
assert not set(train.patient) & set(test.patient)   # no leakage
x, m = load_arrays(train.sample(8, random_state=1), 256)
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for a, img, msk in zip(ax.ravel(), x, m[..., 0]):
    a.imshow(overlay(img, msk, (40, 220, 40))); a.axis("off")

## 2. Train (small config so it finishes in a few minutes)

In [ ]:
from brainseg.config import Config
from brainseg.train import run_training
cfg = Config(); cfg.data.root = ROOT; cfg.train.epochs = 10
summary = run_training(cfg, "../runs/notebook")
summary

## 3. Look at the predictions and curves

In [ ]:
from IPython.display import Image
Image("../runs/notebook/training_curves.png")

In [ ]:
Image("../runs/notebook/test_predictions.png")

## 4. Segment a single slice with the inference API

In [ ]:
from brainseg.predict import Segmenter, read_image
seg = Segmenter.from_path("../runs/notebook/best.keras")
img = read_image(test.image.iloc[0])
res = seg.predict(img)
plt.imshow(overlay(img, res.mask)); plt.axis("off")
print(f"tumour: {res.has_tumor}, area {res.tumor_fraction:.2%}, peak p={res.max_probability:.2f}")